# Cycle 2 - Modèle de régression linéaire avec transformation cible

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from sklearn.pipeline import Pipeline
from agritech_feature_engineering import AgritechFeatureEngineering
from sklearn.model_selection import cross_validate, TimeSeriesSplit
import mlflow
import notebooks.utils as utils
from datetime import datetime as dt
import numpy as np

In [3]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [4]:
dataset = pd.read_csv('../data/rafined/dataset.csv', sep=',', na_values=[''], quotechar='"')

In [5]:
dataset.drop(columns=['Area ISO Code', 'Area ISO Alpha-2 Code', 'Area ISO Alpha-3 Code'], inplace=True)

In [6]:
X_train, X_test, y_train, y_test = utils.test_train_time_split(dataset)

Année de césure calculée : 2009
Proportion réelle train : 78.73% (20349 lignes)
Proportion réelle test  : 21.27% (5499 lignes)


In [7]:
X_train = X_train.sort_values('Year')
y_train = y_train.loc[X_train.index]

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor

numeric_features = [
    'avg_temp',
    'average_rain_fall_mm_per_year',
    'pesticides_tonnes',
    'pesticides_kg_ha',
    'aridity',
    'avg_temp_square',
    'missing_pesticide_value',
    'missing_rain_value',
    'missing_temp_value'
]

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('area_encoding', TargetEncoder(target_type='continuous', random_state=42), ['Area']),
        ('item_encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Item']),
        ('num_pipeline', numeric_transformer, numeric_features),
    ],
    remainder='drop'
)

In [9]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

In [10]:
pipeline = Pipeline([
    ('features', AgritechFeatureEngineering()),
    ('preprocessor', preprocessor),
    ('model', model)
])

In [11]:
# test de log le rendement car la distribution est très étalé
model_log = TransformedTargetRegressor(
    regressor=pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
)

In [12]:
mlflow.set_experiment("Agriculture_optimization")
mlflow.end_run()
mlflow.set_system_metrics_sampling_interval(1)

In [13]:
scoring = {'r2': 'r2', 'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'}

with mlflow.start_run(run_name="Cycle 2 - Régression linéaire avec transformation cible", log_system_metrics=True):

    mlflow.set_tags({
        'date': dt.now().strftime("%Y-%m-%d"),
        'cycle': '2_regression_with_target_log',
        'model_type': 'linear_regression',
        'mlflow.note.content': (
            "Régression linéaire avec transformation cible (log1p) : modèle de base pour prédire le rendement agricole "
            "en utilisant les variables climatiques et agricoles."
        ),
    })

    mlflow.log_params({
        'split': 'temporel',
        'split_year': int(X_test['Year'].min()),
        'model': 'LinearRegression',
        'cv': 'TimeSeriesSplit 5 plis',
        'target_transform': 'log1p',
        'features': 'Area, Item, avg_temp, rain, pesticides, aridity, avg_temp_square, missing_*',
    })

    cv_results = cross_validate(
        model_log,
        X_train,
        y_train,
        cv=TimeSeriesSplit(5),
        scoring=scoring,
        return_train_score=True,
        error_score='raise'
    )

    mlflow.log_metrics({
        'r2': cv_results['test_r2'].mean(),
        'mae': -cv_results['test_mae'].mean(),
        'rmse': -cv_results['test_rmse'].mean(),
        'train_r2': cv_results['train_r2'].mean(),
        'train_mae': -cv_results['train_mae'].mean(),
        'train_rmse': -cv_results['train_rmse'].mean(),
        'fit_time_mean': cv_results['fit_time'].mean()
    })

    print('--- Validation Fold Results ---')
    print(f"Validation R2 : {cv_results['test_r2']}, R2 moyen : {cv_results['test_r2'].mean()}")
    print(f"Validation MAE : {cv_results['test_mae']}, MAE moyen : {cv_results['test_mae'].mean()}")
    print(f"Validation RMSE : {cv_results['test_rmse']}, RMSE moyen : {cv_results['test_rmse'].mean()}")

    print('--- Train Fold Results ---')
    print(f"Train R2 moyen : {cv_results['train_r2'].mean()}")
    print(f"Train MAE moyen : {-cv_results['train_mae'].mean()}")
    print(f"Train RMSE moyen : {-cv_results['train_rmse'].mean()}")

2026/11/19 14:21:54 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/19 14:21:54 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEnco

--- Validation Fold Results ---
Validation R2 : [0.66699664 0.67025604 0.61712061 0.66391182 0.64858691], R2 moyen : 0.6533744027089614
Validation MAE : [-22933.25751827 -23380.9836415  -26709.51099576 -26160.28032735
 -27609.20761412], MAE moyen : -25358.648019397326
Validation RMSE : [-39845.16143319 -39857.07468339 -46154.32987565 -43678.05488071
 -45816.55509238], RMSE moyen : -43070.23519306325
--- Train Fold Results ---
Train R2 moyen : 0.684225622331347
Train MAE moyen : 22346.70861824281
Train RMSE moyen : 38404.912064234675
🏃 View run Cycle 2 - Régression linéaire avec transformation cible at: http://127.0.0.1:5000/#/experiments/1/runs/51c99fe0d6db4c71b5c4417675483b98
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Conclusion du cycle 2 : régression linéaire avec cible en log

Même modèle, mêmes variables et même validation qu'au cycle 1.

Un seul changement : le modèle apprend `log(1 + rendement)` au lieu du rendement brut. Les prédictions sont remises en hg/ha avant le calcul des scores, donc les métriques sont directement comparables à celles du cycle 1.

**Résultats (validation croisée temporelle, 5 plis)** :

| Métrique | Cycle 1 (sans log) | Cycle 2 (log) | Évolution |
|---|---|---|---|
| R² validation | 0.660 | 0.652 | −0.008 |
| MAE validation | 28 350 hg/ha | 25 394 hg/ha | **−10,4 %** |
| RMSE validation | 42 594 hg/ha | 43 128 hg/ha | +1,3 % |
| R² train | 0.678 | 0.684 | +0.006 |

Rappel de la baseline naïve (médiane pays × culture) : R² = 0.836, RMSE = 32 646 hg/ha, MAE = 16 461 hg/ha.

- **Le log améliore l'erreur typique** : la MAE baisse de 10 %. L'erreur moyenne passe d'environ 2,8 à 2,5 tonnes par hectare.
- **Toujours loin de la baseline** : le modèle reste nettement moins précis que la simple médiane pays × culture (MAE 25 394 contre 16 461).

**Bilan** : transformer la cible aide un peu, mais ne règle pas le problème. Une régression linéaire ne peut pas représenter des effets qui changent selon la culture et le pays. La limite vient sûrement du modèle lui-même.

**Piste pour le cycle suivant** : passer à d'autres modèles (Random Forest, Gradient Boosting), capables d'apprendre des effets différents selon la culture et des interactions entre variables.
